**Gradio** est une bibliothèque Python qui permet de créer rapidement une interface web pour un modèle d'IA.

Je l'utilise pour que l'utilisateur charge son fichier et obtienne une prédiction sans écrire de code.

In [1]:
!pip install -q gradio

In [2]:
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

COLONNES = ["type", "complexite", "nb_pieces", "delai"]


def nettoyer_nom(nom):
    """'Complexité ' -> 'complexite' ; 'Nb pièces' -> 'nb_pieces'"""
    nom = unicodedata.normalize("NFKD", str(nom)).encode("ascii", "ignore").decode()
    return nom.strip().lower().replace(" ", "_")


def lire_fichier(fichier):
    chemin = fichier if isinstance(fichier, str) else fichier.name
    if chemin.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(chemin)
    else:
        df = pd.read_csv(chemin, sep=None, engine="python")   # détecte la virgule ou le point-virgule
    df.columns = [nettoyer_nom(c) for c in df.columns]
    return df


def fichier_exemple():
    """Crée un fichier de 1000 demandes fictives (services de la mairie) à télécharger."""
    rng = np.random.default_rng(42)
    # (service, complexité, pièces min, pièces max, délai moyen, écart, poids)
    S = [("extrait naissance/décès", 1, 1, 1, 0, 0, 25), ("copie conforme", 1, 1, 5, 0, 0, 25),
         ("vocation urbanistique", 2, 5, 5, 7, 1, 8), ("certificat de conformité", 3, 5, 5, 30, 4, 6),
         ("enseigne publicitaire", 3, 6, 6, 15, 3, 5), ("autorisation d'installation", 3, 5, 6, 15, 3, 5),
         ("occupation voie publique", 4, 6, 6, 20, 2, 5), ("exploitation local commercial", 4, 6, 7, 30, 5, 6),
         ("permis de bâtir", 4, 9, 9, 21, 3, 7), ("permis de lotir", 5, 8, 8, 45, 6, 3),
         ("permis de démolir", 5, 4, 4, 60, 5, 3), ("régularisation construction", 5, 7, 7, 90, 10, 2)]
    p = np.array([s[6] for s in S], float)
    lignes = [S[i] for i in rng.choice(len(S), 1000, p=p / p.sum())]
    df = pd.DataFrame({
        "type": [r[0] for r in lignes],
        "complexite": [r[1] for r in lignes],
        "nb_pieces": [int(rng.integers(r[2], r[3] + 1)) for r in lignes],
        "delai": [max(0, int(round(r[4] + rng.normal() * r[5]))) for r in lignes],
    })
    df.to_csv("exemple_demandes.csv", index=False)
    return "exemple_demandes.csv"


def entrainer(fichier):
    vide = (None, None, None, gr.update())
    if fichier is None:
        return ("⚠️ Charge d'abord un fichier (ou génère le fichier exemple, puis charge-le).",) + vide
    try:
        df = lire_fichier(fichier)
    except Exception as e:
        return (f"⚠️ Impossible de lire le fichier : {e}",) + vide

    manquantes = [c for c in COLONNES if c not in df.columns]
    if manquantes:
        return (f"⚠️ Colonnes manquantes : {', '.join(manquantes)}. Le fichier doit contenir : "
                f"{', '.join(COLONNES)}. Colonnes trouvées : {', '.join(df.columns)}.",) + vide

    df = df[COLONNES].copy()
    for c in ["complexite", "nb_pieces", "delai"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    avant = len(df)
    df = df.dropna()
    df["type"] = df["type"].astype(str).str.strip()
    if len(df) < 50:
        return (f"⚠️ Il faut au moins 50 demandes valides (il y en a {len(df)}).",) + vide

    X = pd.get_dummies(df[["type", "complexite", "nb_pieces"]], columns=["type"], dtype=int)
    y = df["delai"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

    modeles = {"Régression linéaire": LinearRegression(),
               "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42)}
    lignes, preds = [], {}
    for nom, m in modeles.items():
        m.fit(Xtr, ytr)
        p = m.predict(Xte)
        preds[nom] = p
        lignes.append({"Modèle": nom,
                       "MAE (jours)": round(mean_absolute_error(yte, p), 2),
                       "RMSE (jours)": round(mean_squared_error(yte, p) ** 0.5, 2),
                       "R²": round(r2_score(yte, p), 3)})
    tableau = pd.DataFrame(lignes)
    meilleur = tableau.sort_values("MAE (jours)").iloc[0]["Modèle"]

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.scatter(yte, preds[meilleur], s=18, alpha=0.45, color="#1F6F78")
    lim = [min(yte.min(), preds[meilleur].min()), max(yte.max(), preds[meilleur].max())]
    ax.plot(lim, lim, "--", color="#E9A23B", linewidth=2, label="Prédiction parfaite")
    ax.set_xlabel("Délai réel (jours)")
    ax.set_ylabel("Délai prédit (jours)")
    ax.set_title(meilleur)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.close(fig)

    # modèles finaux : on les réentraîne sur toutes les données pour la prédiction
    finaux = {nom: clone(m).fit(X, y) for nom, m in modeles.items()}
    etat = {"modeles": finaux, "colonnes": list(X.columns), "meilleur": meilleur}
    types = sorted(df["type"].unique())
    note = f" ({avant - len(df)} lignes incomplètes ignorées)" if avant != len(df) else ""
    statut = (f"✅ {len(df)} demandes lues{note} : {len(Xtr)} pour apprendre, {len(Xte)} pour tester. "
              f"**Meilleur modèle : {meilleur}.**")
    return statut, tableau, fig, etat, gr.update(choices=types, value=types[0])


def predire(etat, type_demande, complexite, nb_pieces):
    if not etat:
        return "⚠️ Entraîne d'abord les modèles (bouton « 1. Entraîner les modèles »)."
    if not type_demande:
        return "⚠️ Choisis un type de demande."
    ligne = pd.get_dummies(
        pd.DataFrame([{"type": type_demande, "complexite": complexite, "nb_pieces": nb_pieces}]),
        columns=["type"], dtype=int).reindex(columns=etat["colonnes"], fill_value=0)
    res = {nom: max(float(m.predict(ligne)[0]), 0.0) for nom, m in etat["modeles"].items()}
    best = res[etat["meilleur"]]
    detail = "\n".join(f"- {nom} : {v:.1f} jours" for nom, v in res.items())
    return f"### Délai prédit : {best:.1f} jours (environ {round(best)} jours)\n\n{detail}"


with gr.Blocks(title="Prédiction du délai de traitement") as demo:
    gr.Markdown("# Prédiction du délai de traitement des demandes administratives\n"
                "Charge ton fichier de demandes (CSV ou Excel) avec les colonnes : "
                "`type`, `complexite`, `nb_pieces`, `delai`.")
    etat = gr.State(None)
    with gr.Row():
        with gr.Column():
            fichier = gr.File(label="Fichier de demandes (.csv ou .xlsx)", file_types=[".csv", ".xlsx", ".xls"])
            bouton_ex = gr.Button("Pas de fichier ? Générer un fichier exemple")
            fichier_ex = gr.File(label="Fichier exemple (à télécharger, puis à charger ci-dessus)")
            bouton_train = gr.Button("1. Entraîner les modèles", variant="primary")
        with gr.Column():
            statut = gr.Markdown()
            tableau = gr.Dataframe(label="Résultats sur les données de test")
            graphique = gr.Plot(label="Prédit contre réel (meilleur modèle)")
    gr.Markdown("## 2. Prédire le délai d'une nouvelle demande")
    with gr.Row():
        type_dd = gr.Dropdown(label="Type de demande", choices=[])
        cplx = gr.Slider(1, 5, value=3, step=1, label="Complexité (1 à 5)")
        pieces = gr.Slider(1, 20, value=5, step=1, label="Nombre de pièces")
    bouton_pred = gr.Button("Prédire", variant="primary")
    resultat = gr.Markdown()

    bouton_ex.click(fichier_exemple, None, fichier_ex)
    bouton_train.click(entrainer, fichier, [statut, tableau, graphique, etat, type_dd])
    bouton_pred.click(predire, [etat, type_dd, cplx, pieces], resultat)

demo.launch(share=True, debug=False)   # share=True : crée un lien public temporaire


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20c899dc019a3caa1e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
